<a href="https://colab.research.google.com/github/saphixe/ALLM-A2/blob/main/ALLM_A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os

# Install uv if needed
!pip -q install uv

# Clone BALROG only if missing
if not os.path.exists("/content/BALROG"):
    !git clone https://github.com/balrog-ai/BALROG.git

# Install Python 3.10 and create the environment if missing
if not os.path.exists("/content/balrog-venv/bin/python"):
    !uv python install 3.10
    !uv venv /content/balrog-venv --python 3.10

    # Install BALROG
    %cd /content/BALROG
    !uv pip install --python /content/balrog-venv/bin/python -e .
    !uv pip install --python /content/balrog-venv/bin/python "setuptools==80.9.0"
    !/content/balrog-venv/bin/balrog-post-install

%cd /content/BALROG

/content/BALROG


In [12]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
##########################################################                81.2%

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Ollama server process started")

In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
!ollama run qwen2.5:3b "What is 2 + 2?"

In [ ]:
import requests
import time

start = time.perf_counter()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": "What is 6 time 78?",
        "stream": False
    }
)

elapsed = time.perf_counter() - start

print("Response:", response.json()["response"])
print(f"Time: {elapsed:.2f} seconds")

In [ ]:
!curl http://localhost:11434/v1/models

In [ ]:
%cd /content/BALROG

import os
os.environ["OPENAI_API_KEY"] = "ollama"

!/content/balrog-venv/bin/python eval.py \
    agent.type=naive \
    agent.max_image_history=0 \
    agent.max_text_history=16 \
    eval.num_workers=1 \
    eval.num_episodes.babyai=1 \
    eval.max_steps_per_episode=20 \
    envs.names=babyai \
    'tasks.babyai_tasks=["BabyAI-MixedTrainLocal-v0/goto"]' \
    client.client_name=vllm \
    client.model_id=qwen2.5:3b \
    client.base_url=http://localhost:11434/v1